Langsmith


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OpenAI_API_KEY"] = os.getenv("OpenAI_API_KEY")
os.environ["LANGSMITH_TRACING"] =  "true"

### create the data points


In [11]:
from langsmith import Client
from datetime import datetime

now = datetime.now()
client = Client()

dataset=client.create_dataset("RAG Evaluation Dataset",
    description="A dataset for evaluating RAG models with various tasks")
# sample dataset suitable for Langsmith: each line is a JSON object
client.create_examples(dataset_id=dataset.id, examples=[

    {
       
        "input": "Translate the following sentence to French: 'Hello, how are you?'",
        "output": "Bonjour, comment allez-vous ?",
        "metadata": {"task": "translation", "language": "fr", "created_at": now, "source": "synthetic"}
    },
    {
       
        "input": "Summarize the following paragraph:\n'Langsmith is a platform for tracking and evaluating model runs.'",
        "output": "Langsmith tracks and evaluates model runs.",
        "metadata": {"task": "summarization", "created_at": now, "source": "synthetic"}
    },
    {
        
        "input": "Classify the sentiment: 'I love this product, it's fantastic!'",
        "output": "positive",
        "metadata": {"task": "classification", "labels": ["positive", "neutral", "negative"], "created_at": now}
    },
    {
       
        "input": "Extract the main entity from: 'Google announced a new AI chip today.'",
        "output": "Google",
        "metadata": {"task": "entity_extraction", "created_at": now}
    },
    {
       
        "input": "Answer the question: 'What is the capital of France?'",
        "output": "Paris",
        "metadata": {"task": "qa", "created_at": now}
    },
    {
        
        "input": "Normalize the date '10th Jan 2020' to ISO format",
        "output": "2020-01-10",
        "metadata": {"task": "normalization", "created_at": now}
    }
]
)


{'example_ids': ['8c565aed-71db-47f6-abea-10a35bdd9a74',
  '61e1bb80-c999-4030-b53d-e5db7bca500f',
  '6903e1f7-9b8c-4697-9f6f-94cc7a43fae9',
  'accfa1d4-96c5-41fd-bb3c-fdda0ca5f902',
  '6025f112-58cc-4b23-95bf-1c66053d2669',
  '855c1499-9136-4b85-ab62-e380be930077'],
 'count': 6,
 'as_of': '2026-06-19T12:42:41.078204916Z'}

### Define  Metrices (LLM as Judge)

In [ ]:
import openai
from langsmith import wrappers

openai_client=wrappers.OpenAIClient(api_key=os.getenv("OpenAI_API_KEY"))
eval_instructions = """
You are given a dataset of examples, each with an input and an expected output. Your task is to evaluate the model's output against the expected output and provide a score from 0 to 1, where 1 indicates a perfect match and 0 indicates no match. Additionally, provide a brief explanation for your score.
    
"""

def correctness_evaluator(inputs:dict, outputs:dict, reference_outputs:dict) -> bool:
    user_content = f"""YOu are grading the following questions:
    {inputs['question']}
    here is the real answer:{reference_outputs['answer']}    
    here is the model answer:{outputs['answer']}
     response should be a score  0 or 1 and a brief explanation for your score.
    """
    response = openai_client.chat_completion(

        model="gpt-4o",
        messages=[{"role": "system", "content": user_content}],
        max_tokens=150
    )
    
    return response.choices[0].message['content']

In [ ]:
from langsmith.evaluation import evaluate

# Define evaluation functions
def correctness(run, example):
    """Evaluate if the output matches the expected output"""
    user_content = f"""Evaluate the following:
    Input: {example.inputs['input']}
    Expected Output: {example.outputs['output']}
    Model Output: {run.outputs['output']}
    
    Provide a score from 0 to 1 and brief explanation."""
    
    response = openai_client.chat_completion(
        model="gpt-4o",
        messages=[{"role": "system", "content": user_content}],
        max_tokens=150
    )
    
    return {"score": float(response.choices[0].message['content'].split('\n')[0])}

# Run evaluation on the dataset
eval_results = evaluate(
    lambda x: {"output": "model_prediction"},  # Replace with your model prediction function
    data=dataset,
    evaluators=[correctness],
    experiment_prefix="RAG Evaluation"
)

print(eval_results)